# Day 3 Hands-On Laboratory: Reference Solutions
================================================

This notebook contains complete solutions and reference implementations for the Day 3 exercises.

## Problem 1 Solution: Accelerator Luminosity and Runtime Statistics

In [ ]:
import numpy as np

def calculate_luminosity(f, n_b, N_p, sigma_x, sigma_y):
    # Sahoo Eq. 5.188
    numerator = f * n_b * (N_p ** 2)
    denominator = 4 * np.pi * sigma_x * sigma_y
    return numerator / denominator

# 1. LHC proton-proton collision parameters
f_lhc = 11245.0  # Hz
n_b_lhc = 2808
N_p_lhc = 1.15e11
sigma_lhc = 16e-4  # 16 micrometers in cm

lhc_lum = calculate_luminosity(f_lhc, n_b_lhc, N_p_lhc, sigma_lhc, sigma_lhc)
print(f"LHC Luminosity = {lhc_lum:.4e} cm^-2 s^-1")

# 2. Calculate event rate and total events in 15 hours
# 1 barn = 1e-24 cm^2. 80 mb = 80e-3 * 1e-24 = 8e-26 cm^2
sigma_inel = 80e-3 * 1e-24  
event_rate = lhc_lum * sigma_inel  # events per second
hours = 15.0
total_events = event_rate * hours * 3600.0
print(f"Event rate = {event_rate:.4f} Hz (events/sec)")
print(f"Total events in {hours} hours = {total_events:.4e}")

# 3. Run time required for 2% statistical error
# Statistical error on N events is 1/sqrt(N) = 0.02 ==> N = 1 / (0.02^2) = 2500 events
# Target count requested: N_target = 2.5e9 events with 2% error
target_N = 2.5e9
time_seconds = target_N / event_rate
time_hours = time_seconds / 3600.0
print(f"Time required to collect {target_N:.1e} events = {time_hours:.2f} hours ({time_hours/24.0:.2f} days)")

## Problem 2 Solution: Feynman scaling variable $x_F$

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
from ampt_parser import iter_events
from kinematics import rapidity, feynman_x
import matplotlib.pyplot as plt

filepath = "../Data/subsets/ampt_39_sub100.dat"
sqrt_s = 39.0

all_xf = []
mid_rap_xf = []

for header, particles in iter_events(filepath, max_events=100):
    # Select pions
    mask = np.abs(particles['pid']) == 211
    sel = particles[mask]
    if len(sel) == 0:
        continue
    
    # Feynman x: 2 * pz / sqrt_s
    xf = feynman_x(sel['pz'], sqrt_s)
    all_xf.extend(xf)
    
    # Mid-rapidity pions |y| < 0.5
    y = rapidity(sel['px'], sel['py'], sel['pz'], sel['mass'])
    mid_rap_mask = np.abs(y) < 0.5
    mid_rap_xf.extend(xf[mid_rap_mask])

# Plotting histograms
fig, ax = plt.subplots(figsize=(8, 6))
bins = np.linspace(-0.2, 0.2, 41)

ax.hist(all_xf, bins=bins, color='gray', alpha=0.6, label='All pions', edgecolor='black', linewidth=0.5)
ax.hist(mid_rap_xf, bins=bins, color='red', alpha=0.8, label='Mid-rapidity pions (|y| < 0.5)', edgecolor='black', linewidth=0.5)

ax.set_xlabel(r'Feynman Scaling Variable $x_F$', fontsize=14)
ax.set_ylabel('Raw counts', fontsize=14)
ax.set_title(r'Feynman $x_F$ Distribution for Pions at $\sqrt{s_{NN}} = 39$ GeV', fontsize=14)
ax.legend(frameon=True, fontsize=12)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

## Problem 3 Solution: Parsing AMPT and Invariant Yields

In [ ]:
from kinematics import transverse_momentum

species_info = {
    'pion':   {'pid': 211,  'mass': 0.139570, 'label': r'$\pi^{\pm}$',    'color': 'blue',   'marker': 'o'},
    'kaon':   {'pid': 321,  'mass': 0.493677, 'label': r'$K^{\pm}$',     'color': 'green',  'marker': 's'},
    'proton': {'pid': 2212, 'mass': 0.938272, 'label': r'$p/\bar{p}$',   'color': 'red',    'marker': '^'}
}

y_cut = 0.5
dy = 2 * y_cut
pt_bins = np.linspace(0.0, 3.0, 31)
pt_width = pt_bins[1] - pt_bins[0]
pt_centers = 0.5 * (pt_bins[:-1] + pt_bins[1:])

fig, ax = plt.subplots(figsize=(8, 6))

for name, info in species_info.items():
    all_pt = []
    event_count = 0
    
    for header, particles in iter_events(filepath, max_events=100):
        event_count += 1
        mask = np.abs(particles['pid']) == info['pid']
        sel = particles[mask]
        if len(sel) == 0:
            continue
        
        y = rapidity(sel['px'], sel['py'], sel['pz'], sel['mass'])
        sel = sel[np.abs(y) < y_cut]
        if len(sel) == 0:
            continue
            
        pt = transverse_momentum(sel['px'], sel['py'])
        all_pt.extend(pt)
        
    counts, _ = np.histogram(all_pt, bins=pt_bins)
    inv_yield = counts / (event_count * 2 * np.pi * pt_centers * pt_width * dy)
    inv_yield_err = np.sqrt(counts) / (event_count * 2 * np.pi * pt_centers * pt_width * dy)
    
    valid = counts > 0
    ax.errorbar(pt_centers[valid], inv_yield[valid], yerr=inv_yield_err[valid],
                fmt=info['marker'], color=info['color'], markersize=6, capsize=2,
                label=info['label'])

ax.set_yscale('log')
ax.set_xlabel(r'$p_T$ (GeV/$c$)', fontsize=14)
ax.set_ylabel(r'Invariant Yield $\frac{1}{2\pi p_T} \frac{\mathrm{d}^2N}{\mathrm{d}p_T\mathrm{d}y}$ (GeV/$c$)$^{-2}$', fontsize=14)
ax.set_xlim(0.0, 3.0)
ax.set_ylim(1e-5, 2e2)
ax.legend(frameon=True, fontsize=12)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

## Problem 4 Solution: Boltzmann Fitting and Radial Flow Extraction

In [ ]:
from kinematics import transverse_mass

mt_bins = np.linspace(0.0, 1.5, 31)
mt_width = mt_bins[1] - mt_bins[0]
mt_centers = 0.5 * (mt_bins[:-1] + mt_bins[1:])

teff_results = {}
teff_errors = {}

fig, ax = plt.subplots(figsize=(8, 6))

for name, info in species_info.items():
    all_mt_diff = []
    event_count = 0
    
    for header, particles in iter_events(filepath, max_events=100):
        event_count += 1
        mask = np.abs(particles['pid']) == info['pid']
        sel = particles[mask]
        if len(sel) == 0:
            continue
            
        y = rapidity(sel['px'], sel['py'], sel['pz'], sel['mass'])
        sel = sel[np.abs(y) < y_cut]
        if len(sel) == 0:
            continue
            
        mt = transverse_mass(sel['px'], sel['py'], sel['mass'])
        all_mt_diff.extend(mt - info['mass'])
        
    counts, _ = np.histogram(all_mt_diff, bins=mt_bins)
    mt_val = mt_centers + info['mass']
    inv_yield = counts / (event_count * 2 * np.pi * mt_val * mt_width * dy)
    inv_yield_err = np.sqrt(counts) / (event_count * 2 * np.pi * mt_val * mt_width * dy)

    # Fit range: mT - m0 < 1.0 GeV
    fit_mask = (mt_centers > 0.0) & (mt_centers < 1.0) & (counts > 2)
    X_fit = mt_centers[fit_mask]
    Y_fit = np.log(inv_yield[fit_mask])
    
    slope, intercept = np.polyfit(X_fit, Y_fit, 1)
    teff = -1.0 / slope
    teff_results[name] = teff
    
    # Standard slope error calculation
    residuals = Y_fit - (slope * X_fit + intercept)
    slope_err = np.sqrt(np.sum(residuals**2)/(len(X_fit)-2)) / np.sqrt(np.sum((X_fit - np.mean(X_fit))**2))
    teff_err = slope_err / (slope**2)
    teff_errors[name] = teff_err

    print(f"{info['label']}: Teff = {teff:.4f} +/- {teff_err:.4f} GeV")
    
    valid = counts > 0
    ax.errorbar(mt_centers[valid], inv_yield[valid], yerr=inv_yield_err[valid], 
                fmt=info['marker'], color=info['color'], label=f"{info['label']} (Data)")
    
    X_line = np.linspace(0.0, 1.2, 100)
    Y_line = np.exp(intercept + slope * X_line)
    ax.plot(X_line, Y_line, color=info['color'], linestyle='--', label=f"{info['label']} (Fit)")

ax.set_yscale('log')
ax.set_xlabel(r'$m_T - m_0$ (GeV/$c^2$)', fontsize=14)
ax.set_ylabel(r'Invariant Yield', fontsize=14)
ax.set_xlim(0.0, 1.5)
ax.set_ylim(1e-5, 2e2)
ax.legend(frameon=True, fontsize=10)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

# Teff vs. mass plotting and fitting
masses = np.array([species_info[name]['mass'] for name in teff_results.keys()])
teffs = np.array([teff_results[name] for name in teff_results.keys()])
errs = np.array([teff_errors[name] for name in teff_results.keys()])

slope, intercept = np.polyfit(masses, teffs, 1)
T_th = intercept
v_flow = np.sqrt(2 * slope) if slope > 0 else 0.0

fig, ax = plt.subplots(figsize=(8, 6))
for name in teff_results.keys():
    info = species_info[name]
    ax.errorbar(info['mass'], teff_results[name], yerr=teff_errors[name], 
                fmt=info['marker'], color=info['color'], markersize=10, capsize=4, label=info['label'])

x_fit = np.linspace(0.0, 1.1, 100)
y_fit = intercept + slope * x_fit
ax.plot(x_fit, y_fit, color='black', label=r'Linear Fit: $T_{eff} = T_{th} + \frac{1}{2}m_0\langle v_{flow}\rangle^2$')

ax.set_xlabel(r'Particle Rest Mass $m_0$ (GeV/$c^2$)', fontsize=14)
ax.set_ylabel(r'Effective Temperature $T_{eff}$ (GeV)', fontsize=14)
ax.set_title(r'Slope Parameter $T_{eff}$ vs. Mass', fontsize=14)
ax.set_xlim(0.0, 1.1)
ax.set_ylim(0.05, 0.45)
ax.text(0.05, 0.9, f"Extracted T_th = {T_th:.3f} GeV\nExtracted <v_flow> = {v_flow:.3f} c", 
        transform=ax.transAxes, fontsize=12, bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
ax.legend(frameon=True, fontsize=12)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

## Problem 5 Solution: Event-by-event Multiplicity Correlation: Van Hove Signature

In [ ]:
from ampt_parser import is_charged
from kinematics import pseudorapidity

file_7_7 = "../Data/subsets/ampt_7.7_sub100.dat"
file_39 = "../Data/subsets/ampt_39_sub100.dat"

fig, ax = plt.subplots(figsize=(8, 6))

for filepath, energy_val, col in zip([file_7_7, file_39], [7.7, 39.0], ['red', 'blue']):
    event_pt_means = []
    event_multiplicities = []
    
    for header, particles in iter_events(filepath, max_events=100):
        charged_mask = np.array([is_charged(pid) for pid in particles['pid']])
        sel = particles[charged_mask]
        if len(sel) == 0:
            continue
            
        eta = pseudorapidity(sel['px'], sel['py'], sel['pz'])
        mid_eta_mask = np.abs(eta) < 0.5
        sel = sel[mid_eta_mask]
        
        nch = len(sel)
        if nch < 2:
            continue
            
        pt = transverse_momentum(sel['px'], sel['py'])
        event_pt_means.append(np.mean(pt))
        event_multiplicities.append(nch / 1.0) # dNch/deta
        
    event_pt_means = np.array(event_pt_means)
    event_multiplicities = np.array(event_multiplicities)

    if energy_val == 7.7:
        mult_bins = np.arange(0, 41, 4)
    else:
        mult_bins = np.arange(0, 101, 8)
        
    bin_centers = 0.5 * (mult_bins[:-1] + mult_bins[1:])
    bin_means = []
    bin_errs = []
    
    for i in range(len(mult_bins) - 1):
        low, high = mult_bins[i], mult_bins[i+1]
        bin_mask = (event_multiplicities >= low) & (event_multiplicities < high)
        bin_pt = event_pt_means[bin_mask]
        
        if len(bin_pt) > 1:
            bin_means.append(np.mean(bin_pt))
            bin_errs.append(np.std(bin_pt) / np.sqrt(len(bin_pt)))
        else:
            bin_means.append(np.nan)
            bin_errs.append(np.nan)
            
    bin_centers = np.array(bin_centers)
    bin_means = np.array(bin_means)
    bin_errs = np.array(bin_errs)
    
    valid = ~np.isnan(bin_means)
    ax.errorbar(bin_centers[valid], bin_means[valid], yerr=bin_errs[valid],
                fmt='o-', color=col, linewidth=2, label=f'AMPT \u221as_NN = {energy_val} GeV')

ax.set_xlabel(r'Multiplicity Density $\mathrm{d}N_{\mathrm{ch}}/\mathrm{d}\eta$', fontsize=14)
ax.set_ylabel(r'Average Transverse Momentum $\langle p_T \rangle$ (GeV/$c$)', fontsize=14)
ax.set_title(r'Van Hove Signature: $\langle p_T \rangle$ vs. Multiplicity Density', fontsize=14)
ax.set_xlim(0, 100)
ax.set_ylim(0.25, 0.55)
ax.legend(frameon=True, fontsize=12)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()